# Auditorio ODEON — Pyroomacoustics, simulación verificada

Reproducción independiente con **10 paredes, 25 micrófonos, absorción energética real 0,10 y fuente OMNI**. Primero comprueba la geometría y el tiempo teórico de reverberación; después ejecuta la RIR y compara los 25 receptores con ODEON/I-Simpa.

**Verificación crítica:** el notebook imprime `room.rt_args` y comprueba que la duración real de las RIR no exceda significativamente el límite configurado. No requiere subir archivos: los vértices y posiciones están incorporados.

**Correção confirmada por teste A/B:** chamar `set_ray_tracing` sempre **antes** de `add_microphone_array`. Na implementação anterior, a ordem inversa produziu caudas artificiais longas e T30 próximo de 9,5 s. A reprodução controlada desta versão com 25 microfones resultou em T30 médio de 3,77 s.

In [ ]:
%pip -q install "pyroomacoustics==0.10.1" numpy scipy pandas matplotlib

In [ ]:
import numpy as np, pandas as pd, pyroomacoustics as pra
import matplotlib.pyplot as plt, time, io, json, zipfile
from pathlib import Path
from scipy.signal import butter, sosfiltfilt
from scipy.stats import linregress

# Es recomendable ejecutar 3.000 primero; luego elevar a 50.000.
N_RAYS = 3000
SEED = 123
FS = 8000
ABSORPTION = 0.10
SCATTERING = 0.0
MAX_ORDER = 2
RADIUS = 0.31
TIME_THRES = 8.0
HIST_BIN = 0.01
SOURCE = [3.0, 0.0, 4.0]
OUT = Path('/content/pyroom_odeon_CORREGIDO')
OUT.mkdir(exist_ok=True)
print('Pyroomacoustics',pra.__version__)

In [ ]:
OBJ_TEXT = '# ODEON Example.par, coordinates in metres\nv 0.0 -5.0 0.0\nv 0.0 -5.0 7.5\nv 0.0 5.0 0.0\nv 0.0 5.0 7.5\nv 6.0 -8.0 0.0\nv 6.0 -8.0 10.0\nv 6.0 8.0 0.0\nv 6.0 8.0 10.0\nv 22.0 -8.0 2.56\nv 22.0 -8.0 10.0\nv 22.0 8.0 2.56\nv 22.0 8.0 10.0\ng 1001_Piso_palco\nf 7 5 1 3\ng 1002_Piso_plateia\nf 7 11 9 5\ng 2001_Parede_frontal\nf 3 1 2 4\ng 2002_Parede_palco_norte\nf 3 4 8 7\ng 2002_Parede_palco_sul\nf 1 5 6 2\ng 2003_Parede_plateia_norte\nf 8 12 11 7\ng 2003_Parede_plateia_sul\nf 9 10 6 5\ng 2004_Parede_traseira\nf 11 12 10 9\ng 3001_Teto_palco\nf 4 2 6 8\ng 3002_Teto_plateia\nf 6 10 12 8\n'
RECEIVERS_CSV = 'receptor,x_m,y_m,z_m\nR01,8.0,-5.0,1.52\nR02,8.0,-2.0,1.52\nR03,8.0,1.0,1.52\nR04,8.0,4.0,1.52\nR05,8.0,7.0,1.52\nR06,10.962,-5.0,1.994\nR07,10.962,-2.0,1.994\nR08,10.962,1.0,1.994\nR09,10.962,4.0,1.994\nR10,10.962,7.0,1.994\nR11,13.925,-5.0,2.468\nR12,13.925,-2.0,2.468\nR13,13.925,1.0,2.468\nR14,13.925,4.0,2.468\nR15,13.925,7.0,2.468\nR16,16.887,-5.0,2.942\nR17,16.887,-2.0,2.942\nR18,16.887,1.0,2.942\nR19,16.887,4.0,2.942\nR20,16.887,7.0,2.942\nR21,19.849,-5.0,3.416\nR22,19.849,-2.0,3.416\nR23,19.849,1.0,3.416\nR24,19.849,4.0,3.416\nR25,19.849,7.0,3.416\n'
REFERENCE_CSV = 'receptor,T30_ODEON_s,T30_iSimpa_s,EDT_ODEON_s,EDT_iSimpa_s,C80_ODEON_dB,C80_iSimpa_dB\nR01,3.33,3.741,3.38,3.735,-2.1,-3.291\nR02,3.33,3.723,3.38,3.732,-2.3,-3.753\nR03,3.31,3.72,3.33,3.782,-2.5,-3.826\nR04,3.32,3.724,3.4,3.746,-2.3,-3.48\nR05,3.33,3.716,3.35,3.628,-2.0,-2.752\nR06,3.3,3.735,3.37,3.669,-2.4,-3.549\nR07,3.33,3.719,3.28,3.727,-3.0,-4.312\nR08,3.31,3.716,3.24,3.71,-3.0,-4.177\nR09,3.33,3.737,3.29,3.713,-2.4,-3.566\nR10,3.32,3.703,3.3,3.656,-2.7,-3.533\nR11,3.28,,3.33,,-2.1,\nR12,3.31,3.761,3.24,3.661,-2.4,-3.615\nR13,3.34,3.763,3.24,3.64,-2.6,-3.541\nR14,3.33,3.758,3.17,3.614,-2.3,-3.3\nR15,3.32,3.729,3.24,3.602,-2.3,-3.222\nR16,3.3,3.778,3.22,3.608,-1.8,-2.698\nR17,3.33,3.801,3.21,3.636,-1.7,-2.835\nR18,3.29,3.786,3.15,3.58,-1.9,-3.674\nR19,3.32,3.78,3.2,3.568,-1.5,-2.574\nR20,3.33,3.763,3.18,3.636,-1.5,-2.527\nR21,3.29,3.803,3.31,3.713,-1.5,-2.447\nR22,3.3,3.792,3.24,3.619,-1.2,-3.203\nR23,3.31,3.792,3.16,3.605,-1.4,-3.12\nR24,3.33,3.796,3.13,3.631,-1.4,-3.125\nR25,3.34,3.773,3.34,3.668,-1.0,-2.273\n'
vertices=[]; faces=[]; face_names=[]; current_name='SinNombre'
for line in OBJ_TEXT.splitlines():
    parts=line.split()
    if not parts: continue
    if parts[0]=='v': vertices.append([float(x) for x in parts[1:]])
    elif parts[0]=='g': current_name=parts[1]
    elif parts[0]=='f':
        faces.append([int(x)-1 for x in parts[1:]])
        face_names.append(current_name)
vertices=np.array(vertices, dtype=float)
microphones=pd.read_csv(io.StringIO(RECEIVERS_CSV))
reference=pd.read_csv(io.StringIO(REFERENCE_CSV))
assert len(faces)==10 and len(microphones)==25
print('Geometría:',len(vertices),'vértices,',len(faces),'paredes,',len(microphones),'receptores')

In [ ]:
# Vista 3D de la geometría SIN iniciar cálculo acústico.
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
fig = plt.figure(figsize=(10,7))
ax = fig.add_subplot(111, projection='3d')
ax.add_collection3d(Poly3DCollection([vertices[f] for f in faces], facecolor='lightskyblue', alpha=.16, edgecolor='black'))
ax.scatter(*SOURCE, marker='*', s=170, label='P1 OMNI')
ax.scatter(microphones.x_m, microphones.y_m, microphones.z_m, marker='o', s=18, label='25 receptores')
ax.set(xlabel='X (m)', ylabel='Y (m)', zlabel='Z (m)', xlim=(0,22), ylim=(-8,8), zlim=(0,10))
ax.view_init(elev=23,azim=-65); ax.legend(); plt.show()

In [ ]:
# RECONSTRUCCIÓN EXPLÍCITA: la absorción usada por el motor se verificará inmediatamente.
np.random.seed(SEED)
walls=[pra.wall_factory(vertices[f].T, np.array([ABSORPTION]), np.array([SCATTERING]), name=name)
       for f,name in zip(faces,face_names)]
room=pra.Room(walls, fs=FS, max_order=MAX_ORDER, ray_tracing=True, air_absorption=False)
room.set_ray_tracing(n_rays=N_RAYS, receiver_radius=RADIUS, energy_thres=1e-7,
                     time_thres=TIME_THRES, hist_bin_size=HIST_BIN)
room.add_source(SOURCE)
room.add_microphone_array(microphones[['x_m','y_m','z_m']].to_numpy().T)
wall_table=pd.DataFrame([{'wall':w.name,'area_m2':w.area(), 'absorption':float(w.absorption[0]),
                           'scattering':float(w.scatter[0]) if hasattr(w,'scatter') else SCATTERING}
                          for w in room.walls])
print(wall_table.to_string(index=False))
assert np.allclose([w.absorption[0] for w in room.walls],ABSORPTION)
print('VOLUMEN',float(room.volume),'m³')
print('SABINE',room.rt60_theory('sabine'),'s / EYRING',room.rt60_theory('eyring'),'s')
print('PARÁMETROS REALES DE RT:',room.rt_args)
assert room.rt_args['time_thres']==TIME_THRES
assert room.rt_args['n_rays']==N_RAYS

In [ ]:
t0=time.perf_counter()
room.compute_rir()
duration=time.perf_counter()-t0
print(f'Compute RIR: {duration:.2f} s')
rir_lengths=[len(room.rir[i][0])/FS for i in range(25)]
print('RIR mín',min(rir_lengths),'RIR máx',max(rir_lengths),'s; límite RT',TIME_THRES,'s')
if max(rir_lengths)>TIME_THRES + 0.10:
    raise RuntimeError('La RIR excede el límite time_thres: revisar configuración del simulador.')


In [ ]:
# Análisis independiente de la octava 1000 Hz.
sos=butter(4,[1000/np.sqrt(2),1000*np.sqrt(2)],btype='bandpass',fs=FS,output='sos')
def acoustic_metrics(h,xyz):
    filtered=sosfiltfilt(sos, h).astype(float)
    e=filtered**2
    edc=np.cumsum(e[::-1])[::-1]
    db=10*np.log10(np.maximum(edc/edc[0],1e-30))
    tt=np.arange(len(e))/FS
    def fit(upper,lower):
        m=(db<=upper)&(db>=lower)
        if m.sum()<10:return np.nan,np.nan
        slope,intercept,r,p,se=linregress(tt[m],db[m])
        return (-60/slope if slope<0 else np.nan),r*r
    t30,r2=fit(-5,-35)
    edt,_=fit(-0.1,-10)
    time_direct=np.linalg.norm(np.array(xyz)-np.array(SOURCE))/room.c
    cut=int((time_direct+0.080)*FS)
    c80=10*np.log10(e[:cut].sum()/max(e[cut:].sum(),1e-300))
    return {'T30_PRA_s':t30,'EDT_PRA_s':edt,'C80_PRA_dB':c80,
            'energy_rel_dB':10*np.log10(e.sum()),'R2_T30':r2,'RIR_duration_s':len(h)/FS}
rows=[];rir_dict={}
for i,row in microphones.iterrows():
    h=np.asarray(room.rir[i][0],dtype=np.float32)
    rir_dict[row.receptor]=h
    xyz=[row.x_m,row.y_m,row.z_m]
    rows.append({'receptor':row.receptor,**acoustic_metrics(h,xyz)})
df=pd.DataFrame(rows).merge(reference,on='receptor',how='left')
df['SPL_rel_centrado_dB']=df.energy_rel_dB-df.energy_rel_dB.mean()
print(df[['T30_PRA_s','EDT_PRA_s','C80_PRA_dB','RIR_duration_s']].describe().round(3))
print('Comparación de medias:',df[['T30_PRA_s','T30_ODEON_s','T30_iSimpa_s',
                                   'C80_PRA_dB','C80_ODEON_dB','C80_iSimpa_dB']].mean().round(3))

In [ ]:
# Curvas y gráfico; las RIR originales exportadas también estarán en el ZIP.
fig,ax=plt.subplots(figsize=(11,5))
for idx in [0,12,22]:
    h=rir_dict[microphones.iloc[idx].receptor]
    e=sosfiltfilt(sos,h).astype(float)**2
    edc=np.cumsum(e[::-1])[::-1]
    db=10*np.log10(np.maximum(edc/edc[0],1e-20))
    ax.plot((np.arange(len(db))/FS)[::80],db[::80],label=microphones.iloc[idx].receptor)
ax.set(xlabel='Tiempo (s)',ylabel='EDC (dB)',title='Curvas de Schroeder a 1000 Hz',ylim=(-65,2))
ax.grid(alpha=.25);ax.legend();fig.tight_layout();fig.savefig(OUT/'Schroeder_1000Hz.png',dpi=150);plt.show()
fig,ax=plt.subplots(figsize=(11,4))
ax.plot(range(1,26),df.T30_PRA_s,marker='o',label='PRA')
ax.plot(range(1,26),df.T30_ODEON_s,marker='.',label='ODEON')
ax.plot(range(1,26),df.T30_iSimpa_s,marker='.',label='I-Simpa')
ax.set(xlabel='Receptor',ylabel='T30 (s)',title='T30 por receptor',xticks=np.arange(1,26))
ax.grid(alpha=.25);ax.legend();fig.tight_layout();fig.savefig(OUT/'T30_25.png',dpi=150);plt.show()

In [ ]:
# Exportación con metadatos EFECTIVOS, para auditoría.
df.to_csv(OUT/'resultados_25.csv',index=False)
np.savez_compressed(OUT/'rirs_25.npz',**rir_dict)
meta={'pyroomacoustics':pra.__version__,'n_rays':N_RAYS,'random_seed':SEED,'fs_hz':FS,
      'requested_absorption':ABSORPTION,'absorption_by_wall':{w.name:np.asarray(w.absorption).tolist() for w in room.walls},
      'n_mics':25,'room_volume_m3':float(room.volume),'rt60_sabine_s':float(room.rt60_theory('sabine')),
      'rt60_eyring_s':float(room.rt60_theory('eyring')),'rt_args_effective':room.rt_args,
      'rir_duration_s_by_mic':{row.receptor:len(rir_dict[row.receptor])/FS for _,row in microphones.iterrows()},
      'compute_rir_seconds':duration}
(OUT/'configuracion_VERIFICADA.json').write_text(json.dumps(meta,indent=2,ensure_ascii=False))
zipname=Path(f'/content/PRA_ODEON_VERIFICADO_{N_RAYS}_raios.zip')
with zipfile.ZipFile(zipname,'w',zipfile.ZIP_DEFLATED) as z:
    for p in OUT.iterdir():
        if p.is_file():z.write(p,arcname=p.name)
print('ARCHIVO CREADO:',zipname)
# En Colab: desde Files, descarga el ZIP. Opcional: from google.colab import files; files.download(str(zipname))